Purpose: Look at DE gene results from maSigPro (parents & 3 Yg phys categories).<br>
Author: Anna Pardo<br>
Date initiated: Apr. 21, 2026

In [1]:
import pandas as pd
import numpy as np
import json
import os
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

In [2]:
# set directory paths
local = "./DE_results/"
hpc = "./DEGs_results_Jul06/"

In [3]:
def get_degs_file(filepath):
    df = pd.read_csv(filepath,delim_whitespace=True,quotechar='"')
    genes = list(df.index)
    return genes

In [4]:
# get Yf DEGs
yfde = get_degs_file(os.path.join(local,"degs_Yf.txt"))

In [5]:
len(yfde)

1065

In [6]:
# get Ya DEGs
yade = get_degs_file(os.path.join(local,"degs_Ya.txt"))
len(yade)

940

In [7]:
# get the Yg sets
hpcruns = {}
for f in os.listdir(hpc):
    if (not f.endswith("R2.txt")) and (not f.endswith("pval.txt")):
        if f.startswith("Jul06"):
            ID = f.split("degs_")[1].rstrip(".txt")
        else:
            ID = f.split("_3reps")[0]
        hpcruns[ID] = get_degs_file(os.path.join(hpc,f))

In [8]:
hpcruns.keys()

dict_keys(['facultativeCAM_1', 'facultativeCAM_2', 'facultativeCAM_9', 'CAM_2', 'CAM_4', 'C3+CAM_12', 'facultativeCAM_3', 'C3+CAM_2', 'CAM_5', 'CAM_1', 'facultativeCAM_10', 'C3+CAM_10', 'C3+CAM_8', 'C3+CAM_3', 'CAM_9', 'C3+CAM_7', 'facultativeCAM_7', 'C3+CAM_13', 'CAM_7', 'CAM_3', 'facultativeCAM_6', 'CAM_8', 'facultativeCAM_8', 'CAM_10', 'C3+CAM_5', 'C3+CAM_11', 'facultativeCAM_5', 'CAM_6'])

In [9]:
# split into three dicts: one for each physiology
camd = {k:v for k,v in hpcruns.items() if ("facultative" not in k) and ("C3" not in k)}

In [10]:
fcd = {k:v for k,v in hpcruns.items() if "facultative" in k}

In [11]:
c3d = {k:v for k,v in hpcruns.items() if "C3" in k}

In [12]:
from collections import Counter

def genes_in_at_least_x_entries(d, max_x=10):
    """
    d: dictionary of lists {key: [genes, ...]}
    max_x: highest X to report
    
    Returns:
        counts_by_x -> {X: number of genes found in at least X entries}
        gene_freqs   -> Counter of how many entries each gene appears in
    """
    
    # Count how many dictionary entries each gene appears in
    gene_freqs = Counter()
    
    for gene_list in d.values():
        for gene in set(gene_list):   # avoids duplicates within one list
            gene_freqs[gene] += 1

    # Count genes present in at least X entries
    counts_by_x = {
        x: sum(freq >= x for freq in gene_freqs.values())
        for x in range(1, max_x + 1)
    }

    return counts_by_x

In [13]:
genes_in_at_least_x_entries(fcd)

{1: 5356, 2: 2012, 3: 1044, 4: 582, 5: 334, 6: 172, 7: 83, 8: 28, 9: 0, 10: 0}

In [14]:
genes_in_at_least_x_entries(camd)

{1: 10805,
 2: 6200,
 3: 4114,
 4: 2771,
 5: 1851,
 6: 1188,
 7: 718,
 8: 417,
 9: 199,
 10: 82}

In [15]:
genes_in_at_least_x_entries(c3d)

{1: 9022,
 2: 4261,
 3: 2586,
 4: 1677,
 5: 1069,
 6: 647,
 7: 372,
 8: 187,
 9: 59,
 10: 0}

In [27]:
def genelist_in_x(d, max_x=10):
    """
    d: dictionary of lists {key: [genes, ...]}
    max_x: highest X to report
    
    Returns:
        counts_by_x -> {X: number of genes found in at least X entries}
        gene_freqs   -> Counter of how many entries each gene appears in
    """
    
    # Count how many dictionary entries each gene appears in
    gene_freqs = Counter()
    
    for gene_list in d.values():
        for gene in set(gene_list):   # avoids duplicates within one list
            gene_freqs[gene] += 1

    return gene_freqs

In [28]:
camfreqs = genelist_in_x(camd)
c3freqs = genelist_in_x(c3d)
facfreqs = genelist_in_x(fcd)

In [30]:
camgenes_by_x = {
    x: [gene for gene, freq in camfreqs.items() if freq >= x]
    for x in range(1, 11)
}

In [31]:
c3cam_by_x = {
    x: [gene for gene, freq in c3freqs.items() if freq >= x]
    for x in range(1, 11)
}

In [32]:
faccam_by_x = {
    x: [gene for gene, freq in facfreqs.items() if freq >= x]
    for x in range(1, 11)
}

In [34]:
# save each of these as a JSON
with open("./CAM_DEGs_by_x_overlap_July2026.json","w+") as outfile:
    json.dump(camgenes_by_x,outfile)

In [35]:
with open("./facCAM_DEGs_by_x_overlap_July2026.json","w+") as outfile:
    json.dump(faccam_by_x,outfile)

In [36]:
with open("./C3+CAM_DEGs_by_x_overlap_July2026.json","w+") as outfile:
    json.dump(c3cam_by_x,outfile)

In [37]:
# save parental DEG lists
with open("./Ya_DEGs_masigpro_July2026.txt","w+") as outfile:
    for i in yade:
        outfile.write(i+"\n")

In [38]:
with open("./Yf_DEGs_masigpro_July2026.txt","w+") as outfile:
    for i in yfde:
        outfile.write(i+"\n")